In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
orders = pd.read_csv("orders.csv")
orders.head()

FileNotFoundError: [Errno 2] No such file or directory: 'orders.csv'

In [5]:
from google.colab import files
uploaded = files.upload()

Saving orders.csv to orders (1).csv


In [6]:
from google.colab import files
uploaded = files.upload()

Saving users.json to users.json


In [7]:
from google.colab import files
uploaded = files.upload()

Saving restaurants.sql to restaurants.sql


In [8]:
import os
os.listdir()

['.config',
 'orders.csv',
 'drive',
 'users.json',
 'restaurants.sql',
 'orders (1).csv',
 'sample_data']

In [10]:
import pandas as pd

orders = pd.read_csv("orders.csv")
orders.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [11]:
users = pd.read_json("users.json")
users.head()

,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [13]:
import sqlite3

#create temporary database
conn = sqlite3.connect(":memory:")

#read SQL file
with open("restaurants.sql", "r") as f:
  sql_script = f.read()

#execute SQL commands
conn.executescript(sql_script)

#load table into pandas
restaurants = pd.read_sql_query("SELECT * FROM restaurants", conn)

restaurants.head()

,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [14]:
merged1 = orders.merge(users, on="user_id", how ="left")
merged1.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular


In [15]:
final_df = merged1.merge(restaurants, on="restaurant_id", how="left")
final_df.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [16]:
final_df.to_csv("final_food_delivery_dataset.csv",index=False)

In [18]:
import os
os.listdir()

['.config',
 'final_food_delivery_dataset.csv',
 'orders.csv',
 'drive',
 'users.json',
 'restaurants.sql',
 'orders (1).csv',
 'sample_data']

In [19]:
gold = final_df[final_df["membership"] == "Gold"]

city_revenue = gold.groupby("city")["total_amount"].sum()

city_revenue.sort_values(ascending=False)


,total_amount
city,
Chennai,1080909.79
Pune,1003012.32
Bangalore,994702.59
Hyderabad,896740.19


In [20]:
cuisine_avg = final_df.groupby("cuisine")["total_amount"].mean()

cuisine_avg.sort_values(ascending=False)

,total_amount
cuisine,
Mexican,808.021344
Italian,799.448578
Indian,798.466011
Chinese,798.389020


In [22]:
user_total = final_df.groupby("user_id")["total_amount"].sum()

high_spenders = user_total[user_total > 1000]

len(high_spenders)

2544

In [23]:
# Create rating bins
bins = [3.0,3.5,4.0,4.5,5.0]
labels = ["3.0-3.5", "3.6-4.0", "4.1-4.5", "4.6-5.0"]

final_df["rating_range"] = pd.cut(final_df["rating"], bins=bins, labels=labels, include_lowest=True)

rating_revenue = final_df.groupby("rating_range")["total_amount"].sum()

rating_revenue.sort_values(ascending=False)

/tmp/ipython-input-3801369861.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rating_revenue = final_df.groupby("rating_range")["total_amount"].sum()


,total_amount
rating_range,
4.6-5.0,2197030.75
3.0-3.5,2136772.70
4.1-4.5,1960326.26
3.6-4.0,1717494.41


In [24]:
gold = final_df[final_df["membership"] == "Gold"]

city_avg = gold.groupby("city")["total_amount"].mean()

city_avg.sort_values(ascending=False)


,total_amount
city,
Chennai,808.459080
Hyderabad,806.421034
Bangalore,793.223756
Pune,781.162243


In [26]:
# Count of resteraunts per cuisine
rest_count = final_df.groupby("cuisine")["restaurant_id"].nunique()

# Total revenue per cuisine
cuisine_revenue = final_df.groupby("cuisine")["total_amount"].sum()

# Combine into one table
summary = pd.DataFrame({
    "restaurant_count": rest_count,
    "total_revenue": cuisine_revenue
})

summary.sort_values(by=["restaurant_count", "total_revenue"], ascending=[True, False])


,restaurant_count,total_revenue
cuisine,,
Chinese,120,1930504.65
Italian,126,2024203.80
Indian,126,1971412.58
Mexican,128,2085503.09


In [27]:
total_orders = len(final_df)
gold_orders = len(final_df[final_df["membership"] == "Gold"])

percentage = round((gold_orders / total_orders) * 100)
percentage


50

In [28]:
rest_group = final_df.groupby("restaurant_name").agg(
    order_count=("order_id", "count"),
    avg_order_value=("total_amount", "mean")
)

# Filter < 20 orders
small_rests = rest_group[rest_group["order_count"] < 20]

# Sort by avg_order_value descending
small_rests.sort_values("avg_order_value", ascending=False)


KeyError: 'restaurant_name'

In [29]:
final_df.columns

Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name_x', 'name', 'city', 'membership', 'restaurant_name_y',
       'cuisine', 'rating', 'rating_range'],
      dtype='object')

In [30]:
final_df[["restaurant_name_x","restaurant_name_y","name"]].head()

,restaurant_name_x,restaurant_name_y,name
0,New Foods Chinese,Restaurant_450,User_2508
1,Ruchi Curry House Multicuisine,Restaurant_309,User_2693
2,Spice Kitchen Punjabi,Restaurant_107,User_2084
3,Darbar Kitchen Non-Veg,Restaurant_224,User_319
4,Royal Eatery South Indian,Restaurant_293,User_1064


In [31]:
rest_group = final_df.groupby("restaurant_name_x").agg(
    order_count=("order_id", "count"),
    avg_order_value=("total_amount", "mean")
)


# Filter restaurants with less than 20 orders
small_rests = rest_group[rest_group["order_count"] < 20]

# Sort by average order value descending
small_rests.sort_values("avg_order_value", ascending=False)




,order_count,avg_order_value
restaurant_name_x,,
Hotel Dhaba Multicuisine,13,1040.222308
Sri Mess Punjabi,12,1029.180833
Ruchi Biryani Punjabi,16,1002.140625
Sri Delights Pure Veg,18,989.467222
Classic Kitchen Family Restaurant,19,973.167895
...,...,...
Annapurna Tiffins Punjabi,19,621.828947
Darbar Tiffins Non-Veg,18,596.815556
Darbar Restaurant Punjabi,14,589.972857


In [33]:
combo_revenue = final_df.groupby(["membership", "cuisine"])["total_amount"].sum()

combo_revenue.sort_values(ascending=False)


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [34]:
final_df["order_dt"] = pd.to_datetime(final_df["order_date"])
final_df["mnth"] = final_df["order_dt"].dt.month
final_df["qtr"] = final_df["mnth"].apply(lambda x: "Q1" if x<=3 else ("Q2" if x<=6 else ("Q3" if x<=9 else "Q4")))
qtr_rev = final_df.groupby("qtr")["total_amount"].sum()
qtr_rev.sort_values(ascending=False)


/tmp/ipython-input-3347603967.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  final_df["order_dt"] = pd.to_datetime(final_df["order_date"])


,total_amount
qtr,
Q3,2037385.10
Q4,2018263.66
Q1,2010626.64
Q2,1945348.72


In [35]:
len(final_df[final_df["membership"] == "Gold"])

4987

In [36]:
round(final_df[final_df["city"] == "Hyderabad"]["total_amount"].sum())


1889367

In [37]:
final_df["user_id"].nunique()


2883

In [38]:
round(final_df[final_df["membership"] == "Gold"]["total_amount"].mean(), 2)


np.float64(797.15)

In [39]:
len(final_df[final_df["rating"] >= 4.5])

3374

In [41]:
gold = final_df[final_df["membership"] == "Gold"]
top_city = gold.groupby("city")["total_amount"].sum().idxmax()
len(gold[gold["city"] == top_city])


1337

In [42]:
len(final_df)


10000

In [43]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)
